# Notebook 4: Professional Pathway Analysis
How many players go pro? What do they earn? What's the realistic probability from each tier?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

pros  = pd.read_csv('../data/processed/pro_outcomes.csv')
costs = pd.read_csv('../data/processed/costs.csv')
demo  = pd.read_csv('../data/processed/demographics.csv')

In [ ]:
# --- 4A: MLS salary distribution ---
mls = pros[pros['league'] == 'MLS'].copy()

fig, ax = plt.subplots(figsize=(10,4))
labels = mls['notes'].str.extract(r'^(MLS [\w ]+)')[0].fillna(mls['notes'])
ax.barh(labels, mls['salary_usd_mid'], xerr=[
    mls['salary_usd_mid'] - mls['salary_usd_low'],
    mls['salary_usd_high'] - mls['salary_usd_mid']
], color='#27ae60', alpha=0.8, capsize=5)
ax.set_title('MLS Salary Bands by Player Category (2024)', fontsize=13)
ax.set_xlabel('USD')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'${v:,.0f}'))
plt.tight_layout()
plt.savefig('../data/analysis/mls_salary_bands.png', dpi=150)
plt.show()

In [ ]:
# --- 4B: Probability tree — youth player to pro ---
# Based on published estimates and player pool sizes

# ~4M youth players total → ~730k competitive → ~165k elite private →
# ~20k pro academies → ~500 MLS roster spots (for US players)

funnel = pd.DataFrame([
    {'stage': 'All youth soccer players', 'count': 4_000_000, 'tier': 0},
    {'stage': 'Competitive travel (Tier 2)', 'count': 1_200_000, 'tier': 2},
    {'stage': 'ECNL / MLS NEXT (Tier 3)', 'count': 150_000, 'tier': 3},
    {'stage': 'Pro Academies (Tier 4)', 'count': 20_000, 'tier': 4},
    {'stage': 'Signed to any pro contract (US)', 'count': 2_500, 'tier': 5},
    {'stage': 'MLS/NWSL first division', 'count': 600, 'tier': 6},
    {'stage': 'Top European league', 'count': 50, 'tier': 7},
])

funnel['prob_from_start'] = funnel['count'] / funnel['count'].iloc[0] * 100
print(funnel[['stage','count','prob_from_start']].to_string(index=False))
funnel.to_csv('../data/analysis/probability_matrix.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(funnel['stage'][::-1], funnel['count'][::-1], color='#2980b9', alpha=0.85)
ax.set_xscale('log')
ax.set_title('Youth Soccer to Professional: Player Funnel (Log Scale)', fontsize=12)
ax.set_xlabel('Number of Players (log scale)')
for bar, count in zip(bars, funnel['count'][::-1]):
    ax.text(bar.get_width() * 1.05, bar.get_y() + bar.get_height()/2,
            f'{count:,}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('../data/analysis/player_funnel.png', dpi=150)
plt.show()

In [ ]:
# --- 4C: Salary by league comparison ---
leagues_of_interest = ['MLS', 'NWSL', 'USL Championship', 'Bundesliga', 'Premier League', 'La Liga']
league_data = pros[pros['league'].isin(leagues_of_interest)].copy()

fig, ax = plt.subplots(figsize=(12, 5))
league_order = league_data.groupby('league')['salary_usd_mid'].mean().sort_values(ascending=False).index

for i, league in enumerate(league_order):
    row = league_data[league_data['league'] == league]
    lo = row['salary_usd_low'].mean()
    mid = row['salary_usd_mid'].mean()
    hi = row['salary_usd_high'].mean()
    ax.bar(i, mid, color='#3498db', alpha=0.8)
    ax.errorbar(i, mid, yerr=[[mid-lo],[hi-mid]], fmt='none', color='black', capsize=5)

ax.set_xticks(range(len(league_order)))
ax.set_xticklabels(league_order, rotation=15, ha='right')
ax.set_title('Median Salary by League — US Player Context (2024)', fontsize=13)
ax.set_ylabel('USD (mid estimate)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'${v:,.0f}'))
plt.tight_layout()
plt.savefig('../data/analysis/salary_by_league.png', dpi=150)
plt.show()

In [ ]:
# --- 4D: Academy vs College path to pro ---
path_data = pros[pros['year'] == 2024].copy()
path_data['path'] = np.where(path_data['academy_path'] == 'Yes', 'Academy Direct',
                    np.where(path_data['college_path'] == 'Yes', 'College → Pro', 'Mixed/Other'))

path_summary = path_data.groupby(['league','path'])['salary_usd_mid'].mean().unstack().fillna(0)
print('Average salary by path and league:')
print(path_summary)